google authentication

In [ ]:
!pip install gensim
!pip install spacy
!pip install medspacy
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

In [ ]:
from google.cloud import bigquery
project_id = 'ai-on-healthcare'
client = bigquery.Client(project=project_id)


In [ ]:
#Atherosclerosis of native coronary artery
ICD_FILTER = ['41401']


diagnose_query = """
    SELECT *
    FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
    LIMIT 1000000
"""

# Run the query and convert the results to a Pandas DataFrame
df_diagnoses = client.query(diagnose_query).to_dataframe()
df_diagnoses = df_diagnoses[df_diagnoses['ICD9_CODE'].isin(ICD_FILTER)]
subj_filtered = df_diagnoses['SUBJECT_ID'].unique()
print(subj_filtered)

get notes data

In [ ]:
notes_query = """
    SELECT *
    FROM `physionet-data.mimiciii_notes.noteevents`
    LIMIT 10000
"""

# Run the query and convert the results to a Pandas DataFrame
df_notes = client.query(notes_query).to_dataframe()

df_notes_filtered = df_notes[df_notes['SUBJECT_ID'].isin(subj_filtered)]
df_notes_filtered.head(3)

extract entities

In [ ]:
import numpy as np

def tsne_plot(model,words, preTrained=False):
    "Creates and TSNE model and plots it"
    labels = []
    tokens = []

    for word in words:
      if preTrained:
          tokens.append(model[word])
      else:
          tokens.append(model.wv[word])
      labels.append(word)

    tokens = np.array(tokens)
    tsne_model = TSNE(perplexity=50, early_exaggeration=12, n_components=2, init='pca', n_iter=1000, random_state=23)
    new_values = tsne_model.fit_transform(tokens)

    x = []
    y = []
    for value in new_values:
        x.append(value[0])
        y.append(value[1])

    plt.figure(figsize=(16, 16))
    for i in range(len(x)):
        plt.scatter(x[i],y[i])
        plt.annotate(labels[i],
                     xy=(x[i], y[i]),
                     xytext=(5, 2),
                     textcoords='offset points',
                     ha='right',
                     va='bottom')
    plt.show()

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
corpus=[]
for row in range(0, len(df_notes_filtered)):
  str_tokens=[]
  tokens= nlp(df_notes_filtered.iloc[row]['TEXT']).ents
  for i in range(0, len(tokens)):
    str_tokens.append(tokens[i].text)
  corpus.append(list(str_tokens))

corpus = [tok for note in corpus for tok in note]
print(len(corpus))

get pretrainrd model

In [ ]:
import gensim
import gensim.downloader as api
info = api.info()
pretrained_model= api.load("glove-wiki-gigaword-50")

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
model1 = Word2Vec(corpus, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)

In [ ]:
!pip install -q --force-reinstall "spacy>=3.8,<4.0"
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

# Re-aplica el parche del config.cfg
import os, re, importlib, en_core_sci_sm
importlib.reload(en_core_sci_sm)
model_dir = os.path.dirname(en_core_sci_sm.__file__)
pattern = re.compile(r'include_static_vectors\s*=\s*"?(False|True)"?', re.IGNORECASE)
for root, _, files in os.walk(model_dir):
    for f in files:
        if f == "config.cfg":
            p = os.path.join(root, f)
            with open(p) as fh: text = fh.read()
            new_text, n = pattern.subn(lambda m: f"include_static_vectors = {m.group(1).lower()}", text)
            if n and new_text != text:
                with open(p, "w") as fh: fh.write(new_text)
                print("patched:", p)

import spacy
print("spacy:", spacy.__version__)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="spacy")
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

# Parchear el config.cfg antes de cargar
import os, re, importlib
import en_core_sci_sm
model_dir = os.path.dirname(en_core_sci_sm.__file__)
pattern = re.compile(r'include_static_vectors\s*=\s*"?(False|True)"?', re.IGNORECASE)
for root, _, files in os.walk(model_dir):
    for f in files:
        if f == "config.cfg":
            p = os.path.join(root, f)
            with open(p) as fh: text = fh.read()
            new_text, n = pattern.subn(lambda m: f"include_static_vectors = {m.group(1).lower()}", text)
            if n and new_text != text:
                with open(p, "w") as fh: fh.write(new_text)
                print("patched:", p)

In [ ]:
nlp_scispacy = spacy.load("en_ner_bc5cdr_md")

corpus_sci =[]
for row in range(0, len(df_notes_filtered)):
    str_tokens = []
    tokens = nlp_scispacy(df_notes_filtered.iloc[row]['TEXT']).ents
    for i in range(0, len(tokens)):
        str_tokens.append(tokens[i].text)
    corpus_sci.append(list(str_tokens))
corpus_sci_flat = [tok for note in corpus_sci for tok in note]
print(len(corpus_sci_flat))

In [ ]:
model1 = Word2Vec(corpus_sci, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)

In [ ]:
import medspacy
nlp_medspacy = medspacy.load("en_core_sci_sm", disable=["medspacy_pyrush"])

corpus_med = []
for row in range(0, len(df_notes_filtered)):
    str_tokens = []
    tokens = nlp_medspacy(df_notes_filtered.iloc[row]['TEXT']).ents
    for i in range(0, len(tokens)):
        str_tokens.append(tokens[i].text)
    corpus_med.append(list(str_tokens))
corpus_med = [tok for note in corpus_med for tok in note]
print(len(corpus_med))


In [ ]:
model1 = Word2Vec(corpus_sci, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)